# Implicit Full Waveform Inversion (Marmousi I)

<!---
<img src="workflow.png" width="250" height="240">
--->

---
**Schematic of IFWI:**

    coords --> [Implicit Neural Network] --> Physical Parameters --> [FWI]
                       ^                                               |
                       |                                               |
                       ---------(NN's parameters update)----------------


---
---

### Dependency

In [ ]:
import os, sys
sys.path.insert(1, '../codes')  # insert at 1, 0 is the script path (or '' in REPL)

print("----------------")
!python --version
!nvidia-smi
print("----------------")
print("System Version: ", sys.version)

## ======================================================== ##
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from plot_functions import add_colorbar, imagesc

print("PyTorch Version: ", torch.__version__)
print("----------------")
print("torch.cuda.is_available: ",torch.cuda.is_available())
print("----------------")
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# deterministic behavior
torch.manual_seed(3)
torch.cuda.manual_seed_all(3)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(3)
# random.seed(3)
os.environ['PYTHONHASHSEED'] = str(3)

In [ ]:
ckpt_root = "./checkpoints/salt/"   # 你想要的总目录
os.makedirs(ckpt_root, exist_ok=True)

### Load Marmousi I Model

In [ ]:
import pandas as pd
from scipy.ndimage import gaussian_filter

# Load the velocity model and its initial 
vmodel = np.array(pd.read_csv("./hess_salt.csv")) 
v_init = np.array(pd.read_csv("./hess_salt.csv"))
v_init = gaussian_filter(v_init, sigma=50)

dz = 15
nz, nx = vmodel.shape
print("Original Model Shape: {}, Grid Interval: {}m".format(vmodel.shape, dz))

################# Plot true & initial velocity model #################
fig = plt.figure(figsize=(10, 2))
gs = fig.add_gridspec(1, 2)
ax = fig.add_subplot(gs[0, 0])
ax.set(ylabel="Depth z[km]", xlabel="Distance x[km]")
im = ax.imshow(vmodel/1000, extent=[0, nx*dz/1000, nz*dz/1000, 0], aspect=1, cmap='RdBu_r')
cbar = add_colorbar(ax, im, ax.transAxes, width="3%", ctitle='km/s')

ax = fig.add_subplot(gs[0, 1])
ax.set( xlabel="Distance x[km]")
im = ax.imshow(v_init/1000, extent=[0, nx*dz/1000, nz*dz/1000, 0], aspect=1, cmap='RdBu_r')
cbar = add_colorbar(ax, im, ax.transAxes, width="3%", ctitle='km/s')
################# Plot true & initial velocity model #################

sample_interval = 1
dz = dz * sample_interval
dz = 10
vp_tensor = torch.from_numpy(vmodel[None, ::sample_interval, ::1]).type(dtype=torch.float32).to(device)
vi_tensor = torch.from_numpy(v_init[None, ::sample_interval, ::1]).type(dtype=torch.float32).to(device)
nv, nz, nx = vp_tensor.shape
print("Resampled Model Shape: {}, Grid Interval: {}m".format((nz, nx), dz))

# Setting locations of sources and receivers
xs = torch.arange(10, nx, 20, dtype=torch.long).repeat([nv, 1])      # x-coordinate for sources
ns = xs.shape[1]                                                        # number of shots 
xr = torch.arange(0, nx, 1, dtype=torch.long).repeat([nv, ns, 1])       # x-coordinate for receivers
zs = torch.full((nv, ns), 1, dtype=torch.long)                          # depth of sources 
zr = torch.full((nv, ns, nx), 2, dtype=torch.long)                      # depth of receivers
print("Number of shots: {}, with interval: {}m, in depth: {}m".format(ns, (xs[0, 1]-xs[0, 0])*dz, zs[0, 0]*dz))
print(xs)

print(vp_tensor.max(),vp_tensor.min())
VMIN, VMAX = np.percentile(vp_tensor[0].cpu().numpy()/1000, [2, 98])
################# Plot true & initial velocity model #################
fig = plt.figure(figsize=(10, 2))
gs = fig.add_gridspec(1, 2)
ax = fig.add_subplot(gs[0, 0])
ax.set(ylabel="Depth z[km]", xlabel="Distance x[km]")
im = ax.imshow(vp_tensor[0].cpu().numpy()/1000, extent=[0, nx*dz/1000, nz*dz/1000, 0], vmin=1, vmax=4.7,aspect=1, cmap='RdBu_r')
cbar = add_colorbar(ax, im, ax.transAxes, width="3%", ctitle='km/s')

ax = fig.add_subplot(gs[0, 1])
ax.set( xlabel="Distance x[km]")
im = ax.imshow(vi_tensor[0].cpu().numpy()/1000, extent=[0, nx*dz/1000, nz*dz/1000, 0], vmin=1, vmax=4.7, aspect=1, cmap='RdBu_r')
cbar = add_colorbar(ax, im, ax.transAxes, width="3%", ctitle='km/s')

fig.savefig(
    os.path.join(ckpt_root, "Vp_true.png"),
    dpi=300,
    bbox_inches="tight"
)
################# Plot true & initial velocity model #################

### Foward modeling through theory-based RNN

In [ ]:
from rnn_fd import rnn2D
from generator import wGenerator

freeSurface = True                                                      # free surface option for forward modeling
npad = 15                                                               # velocity padding in grid points
freq = 8                                                               # dominant frequency of wavelet in Hz
dt = 0.0010                                                           # time samling interval, fixed for all shots gathers
nt = 2000                                                              # number of samples in time
t = dt * torch.arange(0, nt, dtype=torch.float32)                       # create time vector
wavelet = wGenerator(t, freq).ricker().to(device)                       # generate wavelet
nx_pad = nx + 2 * npad
nz_pad = nz + npad if freeSurface else nz + 2 * npad
f = np.arange(0, nt/2+1) / (nt*dt)

fig = plt.figure(figsize=(10, 2.5))
gs = fig.add_gridspec(1, 2)
ax = fig.add_subplot(gs[0, 0])
ax.set(xlabel="$Time$", ylabel="$Amp$", title="$Ricker$", xlim=[0, 2])
ax.plot(t, wavelet.cpu().numpy(), color='red', linestyle='-', linewidth=1.5)
ax.grid(True, which='both', linestyle='--', color='grey', linewidth=.8, alpha=1.0)
ax.minorticks_on()

# ax = fig.add_subplot(gs[0, 1])
# ax.set(xlabel="$Frequency [Hz]$", ylabel="$Amp$", title="$Amp Spectrum$", xlim=[0, 20])
# ax.plot(f, np.abs(torch.fft.rfft(wavelet).cpu().numpy()), color='red', linestyle='-', linewidth=1.5)
# ax.grid(True, which='both', linestyle='--', color='grey', linewidth=.8, alpha=1.0)
# ax.minorticks_on()

################## Check the stability condition #################
print(vp_tensor.max()*dt/dz/np.sqrt(1/2),"< 1") # should <1
print(vp_tensor.min()/10/freq/dz,"> 1") # should >1

forward_rnn = rnn2D(nz, nx, zs, xs, zr, xr, dz, dt, 
                    npad=npad, order=2, vmax=vp_tensor.max(), 
                    log_para=1e-6,
                    freeSurface=True, 
                    dtype=torch.float32, 
                    device=device).to(device)

# forward modeling
_, _, shots, _ = forward_rnn(vmodel=vp_tensor.to(device), segment_wavelet=wavelet)

In [ ]:

fig=plt.figure(figsize=(ns*1.5, 8))
imagesc(fig,
        shots.cpu().numpy().reshape(-1, ns, nt, nx),
        vmin=-shots.max()/20,
        vmax=shots.max()/20,
        extent=[0, nx*dz/1000, t.numpy().max(), 0],
        aspect=6,
        nRows_nCols=(1, ns),
        cmap='RdBu_r', #seismic
        ylabel="Time (s)",
        xlabel="Position (km)",
        clabel="",
        xticks=np.arange(0., int(nx*dz/1000), 2),
        yticks=np.arange(0., t.numpy().max(), .5),
        fontsize=8,
        cbar_width="7%",
        cbar_height="100%",
        cbar_loc='lower left')
# fig.tight_layout(pad=-0.85)

## 0. IFWI with a pretrained SIREN

Three modes in IFWI2d class: IFWI, FWI, IRN. 

One can perform FWI, or IFWI, or pretraining the network by defining the mode.


In this section, the SIREN will be firstly pretrained with the initial model, and then IFWI will be performed.

pretraining SIREN with initials can be found in IRNN-Marmousi notebook.

In [ ]:
method = "irn"   # 你想要的总目录
save_ckpt=os.path.join(ckpt_root, method)
print(save_ckpt)
os.makedirs(save_ckpt, exist_ok=True)

In [ ]:
import wandb
# wandb.finish()
# wandb.login(relogin=True)
wandb.login()

from ifwi_modules import IFWI2D
ifwi_model = IFWI2D(mean=(vp_tensor/1000).mean(), 
                    std=(vp_tensor/1000).std(),
                    neuron=[2, 128, 128, 128, 128, 1], 
                    omega_0=30, 
                    prob=0.2,
                    activation='sine', 
                    bias=True, 
                    dropout=False,
                    outermost_linear=True,
                    nz=nz,
                    nx=nx,
                    zs=zs,
                    xs=xs,
                    zr=zr, 
                    xr=xr,
                    dz=dz,
                    dt=dt,
                    npad=npad, 
                    order=2, 
                    vmax=vp_tensor.max(),
                    log_para=1e-6,
                    segment_size=len(t),
                    vpadding=None,
                    freeSurface=True,
                    dtype=torch.float32,
                    device=device,
#                     pretrained="ifwi_pretrain_marmousi.pth", ########### the one line difference with the next section.
                    netOpt='IFWI',
                    method = method)

save_file_name = os.path.join(
    save_ckpt,
    "MarmousiI_pretrain_IFWI_PMLvel({:d}x{:d})_dz({:.2f})_nt({:d})_dt({:.4f})_freq({:.1f})_seg({:d})_lr({:.1e})-"
    .format(nz, nx, dz, nt, dt, freq, len(t), 1e-4)
)

print(save_file_name)

In [ ]:
%%time
%%wandb


train_loss_ifwi, vpred = ifwi_model.train(MaxIter=4001, 
                                          vmodel=None,
                                          wavelet=wavelet, 
                                          shots=shots, 
                                          alpha=0,
                                          option=0, 
                                          log_interval=100, 
                                          learning_rate=1e-4,
                                          wandb=wandb,
                                          resume_file_name=None, 
                                          save_file_name=save_file_name)

In [ ]:
print(vpred.shape)

In [ ]:
# after training with 1e-4 with 100 summary interval and max 4001 epochs
save_file_name = "checkpoints/salt/irn/MarmousiI_pretrain_IFWI_PMLvel(187x362)_dz(10.00)_nt(2000)_dt(0.0010)_freq(8.0)_seg(2000)_lr(1.0e-04)-"
vpred, coords = ifwi_model.predict(resume_file_name=save_file_name+'checkpoint-4001.pth', best=True)

fig = plt.figure(figsize=(5, 5))
gs = fig.add_gridspec(1, 1)
ax = fig.add_subplot(gs[0, 0])
ax.set(ylabel="Depth z[km]", xlabel="Distance x[km]", title="Vp")
im = ax.imshow(vpred[:,:150].squeeze().cpu().detach()/1000, vmin=2, vmax=6, extent=[0, nx*dz/1000, nz*dz/1000, 0], aspect=1, cmap='RdBu_r')
cbar = add_colorbar(ax, im, ax.transAxes, width="3%", ctitle='$km/s$')
fig.savefig(
    os.path.join(ckpt_root, "Vp_pre.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)
fig = plt.figure(figsize=(5, 5))
gs = fig.add_gridspec(1, 1)
ax = fig.add_subplot(gs[0, 0])
ax.set(ylabel="Depth z[km]", xlabel="Distance x[km]", title="Vp")
im = ax.imshow(vp_tensor.squeeze().cpu()/1000, vmin=VMIN, vmax=VMAX, extent=[0, nx*dz/1000, nz*dz/1000, 0], aspect=1, cmap='RdBu_r')
cbar = add_colorbar(ax, im, ax.transAxes, width="3%", ctitle='$km/s$')

fig.savefig(
    os.path.join(ckpt_root, "Vp_true.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.close(fig)
plt.show()

In [ ]:
loss_array = np.array(train_loss_ifwi)  # shape: [N, 3]
loss_file = os.path.join(save_ckpt, f"train_loss_history_{method}.txt")
np.savetxt(
    loss_file,
    loss_array,
    fmt="%.6e",
    header="TotalLoss  DataLoss  RegLoss",
    comments=""
)